# Notebook 03 — Hard-Negative Ablation

**Goal:** Evaluate retrieval performance under three negative-sampling strategies of increasing difficulty.

For each evaluation query, we construct a restricted candidate pool of ground-truth APIs + 99 negatives, build a local FAISS index, and measure retrieval quality. This isolates the effect of negative difficulty from corpus size.

**Negative conditions:**
| Strategy | Source | Expected Difficulty |
|----------|--------|--------------------|
| Random | Uniform sample from corpus | Easy |
| Category Siblings | Same ToolBench category as ground truth | Medium |
| DFSDT Failure Paths | APIs considered and rejected during DFSDT execution | Hard |

**Prerequisite:** Run `01_index_apis.ipynb` first.

## Setup

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))
print(f"Repo:      {REPO_ROOT}")
print(f"Toolbench: {TOOLBENCH_DIR}")

In [ ]:
for fname in ['corpus_embeddings.npy', 'api_names.json']:
    path = PROJECT_DIR / fname
    if path.exists():
        print(f"Found {fname}")
    else:
        print(f"WARNING: {fname} not found — run notebook 01 first")

## Load Data

In [ ]:
import json, numpy as np
from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_random_negatives, build_category_sibling_negatives, build_dfsdt_negatives
from models.embeddings import get_embeddings
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import recall_at_k, mean_reciprocal_rank

corpus = load_api_corpus(TOOLBENCH_DIR / "toolenv" / "tools")
lookup = build_api_lookup(corpus)
corpus_matrix = np.load(PROJECT_DIR / "corpus_embeddings.npy")
with open(PROJECT_DIR / "api_names.json") as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}

evals = load_eval_examples(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json")
with open(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json") as f:
    raw_evals = json.load(f)

query_embeddings = get_embeddings([e["user_query"] for e in evals])
print(f"Loaded {len(evals)} eval examples")

## Restricted-Pool Evaluation Framework

For each query, we build a candidate pool of `|A+| + 99` APIs, construct a local FAISS index over just those candidates, and evaluate retrieval within this restricted pool. This setup isolates the difficulty of the negative type from corpus size effects (see Section 4.3 of the paper).

In [ ]:
def evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix,
                             corpus, lookup, name_to_idx,
                             negative_type: str, n_negatives: int = 99, ks=[1, 5, 10]):
    """
    For each eval example:
      1. Build a restricted candidate pool: ground truth + n negatives
      2. Extract sub-matrix of embeddings for that pool
      3. Build a small FAISS index over just those candidates
      4. Retrieve top-k and evaluate
    """
    recall_scores = {k: [] for k in ks}
    mrr_scores = []

    for i, ex in enumerate(evals):
        raw_ex = raw_evals[ex["raw_idx"]]
        gt_names = ex["ground_truth_apis"]
        gt_indices = [name_to_idx[n] for n in gt_names if n in name_to_idx]
        if not gt_indices:
            continue

        # Build negative pool
        if negative_type == "random":
            negatives = build_random_negatives(corpus, gt_names, n=n_negatives)
        elif negative_type == "sibling":
            negatives = build_category_sibling_negatives(corpus, gt_names, lookup, n=n_negatives)
        elif negative_type == "dfsdt":
            negatives = build_dfsdt_negatives(raw_ex, corpus, gt_names, lookup, n=n_negatives)
        else:
            raise ValueError(f"Unknown negative type: {negative_type}")

        # Build candidate pool: positives + negatives
        candidate_names = gt_names + [a["action_name"] for a in negatives]
        candidate_indices = [name_to_idx[n] for n in candidate_names if n in name_to_idx]
        if len(candidate_indices) < 2:
            continue

        candidate_embs = corpus_matrix[candidate_indices].astype(np.float32)
        local_index = build_faiss_index(candidate_embs)

        # Map global GT indices to local indices
        global_to_local = {g: j for j, g in enumerate(candidate_indices)}
        local_gt = [global_to_local[g] for g in gt_indices if g in global_to_local]
        if not local_gt:
            continue

        top_k = retrieve_top_k(query_embeddings[i], local_index, k=min(10, len(candidate_indices)))
        for k in ks:
            recall_scores[k].append(recall_at_k(top_k, local_gt, k))
        mrr_scores.append(mean_reciprocal_rank(top_k, local_gt))

    return {
        **{f"recall@{k}": float(np.mean(recall_scores[k])) for k in ks},
        "mrr": float(np.mean(mrr_scores)),
        "n_examples": len(mrr_scores),
    }

## Run All Three Negative Conditions

We evaluate the same `text-embedding-3-small` baseline under three negative-sampling strategies. The gap between random and structured negatives quantifies how much the model relies on surface-level lexical cues vs. functional understanding.

In [ ]:
print("Running: Random negatives...")
random_results = evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix, corpus, lookup, name_to_idx, "random")

print("Running: Category sibling negatives...")
sibling_results = evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix, corpus, lookup, name_to_idx, "sibling")

print("Running: DFSDT failure-path negatives...")
dfsdt_results = evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix, corpus, lookup, name_to_idx, "dfsdt")

print("\n=== Hard-Negative Ablation Results ===")
print(f"{'Condition':<25} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print("-" * 57)
for label, res in [("Random", random_results), ("Category Siblings", sibling_results), ("DFSDT Failure Paths", dfsdt_results)]:
    print(f"{label:<25} {res['recall@1']:>8.4f} {res['recall@5']:>8.4f} {res['recall@10']:>8.4f} {res['mrr']:>8.4f}")

## Save Results and Visualize

The bar chart below shows Recall@5 across the three conditions, illustrating the ~14-point drop from random to structured negatives.

In [ ]:
import matplotlib.pyplot as plt

all_results = {
    "random": random_results,
    "sibling": sibling_results,
    "dfsdt": dfsdt_results,
}
with open(PROJECT_DIR / "results_hard_negatives.json", "w") as f:
    json.dump(all_results, f, indent=2)

# Bar chart: Recall@5 across conditions
labels = ["Random", "Category\nSiblings", "DFSDT\nFailure Paths"]
r_at_5 = [random_results["recall@5"], sibling_results["recall@5"], dfsdt_results["recall@5"]]
colors = ["steelblue", "darkorange", "firebrick"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, r_at_5, color=colors, edgecolor="white")
ax.bar_label(bars, fmt="{:.3f}", padding=3)
ax.set_ylabel("Recall@5")
ax.set_title("Retrieval Difficulty by Negative Pool Type\n(OpenAI text-embedding-3-small baseline)")
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(PROJECT_DIR / "recall_at_5_ablation.png", dpi=150)
plt.show()
print("Saved results and figure")